# S23D inference

Generate 3D conformers from SMILES with the pretrained S23D-B model from Hugging
Face and view them. Install `smiles23d` as described in the
[README](../README.md#installation), then run this notebook in Jupyter.

In [ ]:
import py3Dmol
import torch
from rdkit import Chem
from rdkit.Chem import rdDepictor, rdMolAlign

from smiles23d.huggingface import S23DDiffusionHF

## Load a pretrained model

The weights download from Hugging Face on first use. The paper samples with
300 diffusion steps. For the smaller S23D-S model, load `ibm-research/S23D.8m.v1` instead.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = S23DDiffusionHF.from_pretrained("ibm-research/S23D.25m.v1", map_location=device)
model.eval()
model.set_predict_parameters(kind="VPSDE", n_steps=300)

## Generate conformers

`predict` takes SMILES strings or RDKit molecules, alone or in a list, and adds
hydrogens, since the model was trained with them. The result has one entry per
input molecule, in input order: a list of RDKit molecules with one conformer each.

In [ ]:
smiles = {
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}
conformers = model.predict(list(smiles.values()), samples_per_molecule=8)
results = dict(zip(smiles, conformers))
for name, samples in results.items():
    print(f"{name}: {len(samples)} conformers with {samples[0].GetNumAtoms()} atoms")

## View the molecules

Each row shows the input structure in 2D, then four of its generated conformers.
Drag to rotate, scroll to zoom.

In [ ]:
for name, samples in results.items():
    print(name)
    flat = Chem.AddHs(Chem.MolFromSmiles(smiles[name]))
    rdDepictor.Compute2DCoords(flat)
    view = py3Dmol.view(viewergrid=(1, 5), width=950, height=190, linked=False)
    view.addModel(Chem.MolToMolBlock(flat), "mol", viewer=(0, 0))
    for col, mol in enumerate(samples[:4], start=1):
        view.addModel(Chem.MolToMolBlock(mol), "mol", viewer=(0, col))
    view.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}})
    view.zoomTo()
    view.show()

All eight ibuprofen conformers without hydrogens, aligned on their heavy atoms,
one color per conformer:

In [ ]:
samples = results["Ibuprofen"]
overlay = Chem.Mol(samples[0])
for mol in samples[1:]:
    overlay.AddConformer(mol.GetConformer(), assignId=True)
overlay = Chem.RemoveHs(overlay)
rdMolAlign.AlignMolConformers(overlay)

colors = ["blue", "orange", "green", "red", "purple", "brown", "magenta", "teal"]
view = py3Dmol.view(width=450, height=450)
for i, (conf, color) in enumerate(zip(overlay.GetConformers(), colors)):
    view.addModel(Chem.MolToMolBlock(overlay, confId=conf.GetId()), "mol")
    view.setStyle({"model": i}, {"stick": {"color": color}})
view.zoomTo()
view.show()

## Optional: start from RDKit molecules

`predict` also takes RDKit molecules, for example ones you have already prepared.
It adds hydrogens to these too, so calling `Chem.AddHs` first is optional.

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in smiles.values()]
from_mols = model.predict(mols, samples_per_molecule=8)
for name, samples in zip(smiles, from_mols):
    print(f"{name}: {len(samples)} conformers with {samples[0].GetNumAtoms()} atoms")